# Program lifecycle via Debug Mailbox on MCXC353

The MCXC353 device exposes a dedicated Debug Mailbox command - `program-lifecycle` - that requests the device to advance its lifecycle state to a target value. The set of named lifecycle values is published by SPSDK in the device's CMPA Life Cycle register specification (`cmpa_lc.json`) and is the single source of truth used by both the PFR tooling and the Debug Mailbox command.

The device only accepts the `program-lifecycle` command while it is still in the initial `DEVELOP` lifecycle state. Once the new value has been accepted, the device performs an internal reset to apply the transition, which is **irreversible**. After the reset, the new lifecycle state becomes active and any subsequent attempt to re-issue `program-lifecycle` will be rejected by the device.

From SPSDK's perspective, the flow is:

- Open the Debug Mailbox over the chosen probe interface.
- Resolve the requested target name (e.g. `IN_FIELD1`) to the 32-bit encoded value from `cmpa_lc.json`.
- Issue the `program-lifecycle` command with that value.
- Treat the post-command timeout caused by the device-side reset as a successful transition.

## WARNING!

**This operation permanently advances the device lifecycle and cannot be undone.** Each lifecycle transition narrows the set of allowed debug operations. Run this notebook only on a development board you are prepared to lose unrestricted debug access to.

## 1. Prerequisites

- SPSDK installed with the examples extension: `pip install spsdk[examples]` (see the [installation](../../_knowledge_base/installation_guide.rst) guide).
- An MCXC353 board connected via a supported debug probe (J-Link in this example).
- The device is currently in the `DEVELOP` lifecycle state. The command will be rejected by the device in any other state.

### 1.1 Known lifecycle values

The values below come from the MCXC353 CMPA Life Cycle specification. Normal customer transitions accepted by SPSDK are `IN_FIELD1`, `IN_FIELD2`, and `IN_FIELD_LOCKED`.

| Name               | 32-bit value | Description                                                              |
|--------------------|-------------:|--------------------------------------------------------------------------|
| `DEVELOP`          | `0x9635FC03` | Initial customer development state; not a transition target              |
| `IN_FIELD1`        | `0x9635F807` | In-field state with password debug authentication                        |
| `IN_FIELD2`        | `0x9635F00F` | In-field state with debug access disabled                                |
| `IN_FIELD_LOCKED`  | `0x963530CF` | Permanently locked in-field state                                        |
| `BRICKED`          | `0x9635A55A` | Permanently disables the device; requires the CLI `--force` option        |
| `NXP_FIELD_RETURN` | `0x9635E01F` | NXP-internal value; not a supported customer transition target           |

The notebook uses the **name** form throughout. SPSDK resolves names through `cmpa_lc.json`, validates the target against the MCXC353 policy, and requires `--force` for destructive or raw device-specific values.

### 1.2 Prepare the environment

In [1]:
import os

# Colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

# Setup workspace
WORKSPACE = "workspace/"
VERBOSITY = "-v"

os.makedirs(WORKSPACE, exist_ok=True)

# Device configuration
FAMILY = "mcxc353"
INTERFACE = "jlink"

# Target lifecycle as a known name; change to any value listed in section 1.1.
# Passing a numeric value (e.g. "0x9635F807") is also supported.
TARGET_LIFECYCLE = "IN_FIELD1"

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


### 1.3 Verify the debug probe is detected

Use `nxpdebugmbox` to list connected probes.

In [2]:
%! nxpdebugmbox $VERBOSITY -i $INTERFACE cmd -f $FAMILY start

nxpdebugmbox -v -i jlink cmd -f mcxc353 start 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
Start Debug Mailbox succeeded


## Step 1: Inspect the current lifecycle

Before any irreversible operation, confirm that the device is still in `DEVELOP`. The `get-crp` command reports the CRP / lifecycle level reported by the device.

In [3]:
%! nxpdebugmbox $VERBOSITY -i $INTERFACE -n cmd -f $FAMILY get-crp

nxpdebugmbox -v -i jlink -n cmd -f mcxc353 get-crp 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
CRP level is: DEVELOP.
Get CRP Level succeeded


## Step 2: Program the target lifecycle

Issue the `program-lifecycle` command. The `-y` flag suppresses the interactive confirmation prompt - omit it to be asked for confirmation before the transition is applied.

In [4]:
%! nxpdebugmbox $VERBOSITY -i $INTERFACE cmd -f $FAMILY program-lifecycle -l $TARGET_LIFECYCLE -y

nxpdebugmbox -v -i jlink cmd -f mcxc353 program-lifecycle -l IN_FIELD1 -y 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
Program lifecycle IN_FIELD1 (0x9635f807) succeeded


## Step 3: Verify the new lifecycle

After the device-side reset, re-open the Debug Mailbox and read the lifecycle again. The reported value must match the target programmed in Step 2.

In [5]:
%! nxpdebugmbox $VERBOSITY -i $INTERFACE cmd -f $FAMILY start
%! nxpdebugmbox $VERBOSITY -i $INTERFACE -n cmd -f $FAMILY get-crp

nxpdebugmbox -v -i jlink cmd -f mcxc353 start 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
Start Debug Mailbox succeeded
nxpdebugmbox -v -i jlink -n cmd -f mcxc353 get-crp 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
CRP level is: IN_FIELD1.
Get CRP Level succeeded


## Summary

This notebook demonstrated programming the MCXC353 device lifecycle via the Debug Mailbox `program-lifecycle` command:

1. Verified the probe connection and current lifecycle.
2. Issued `program-lifecycle` with a named target value resolved from `cmpa_lc.json`.
3. Confirmed the new lifecycle after the device reset.

Because the operation is irreversible, integrate it into your production flow only after validating the target value against your security policy.